# Capítulo 9 — Avaliação do sistema (System Evaluation)

**Livro:** começa na página **426** do PDF.

📗 **Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")  # ia.py (modelos via OpenRouter) e corpus.py (base de conhecimento de exemplo)
from ia import *
from corpus import DOCS
import re, json, math, random
import numpy as np
print("Pronto. Modelo padrão:", MODELO, "| documentos na base:", len(DOCS))

## Evaluation system for RAG systems · PDF p. 428

#### 🧪 Exercício: Métricas de recuperação: hit rate e MRR

Monte um pequeno conjunto de avaliação (6 perguntas → id correto) e calcule *hit rate@3* e MRR do BM25. Sem modelo.

In [ ]:
from rank_bm25 import BM25Okapi
bm = BM25Okapi([re.findall(r"\w+", (d["titulo"] + " " + d["texto"]).lower()) for d in DOCS])
avaliacao = [("que dia a biblioteca fecha?", "d01"), ("preciso de comprovante de endereço?", "d02"), ("posso renovar o empréstimo?", "d03"),
             ("quanto é a multa por dia?", "d04"), ("tem livros em braile?", "d10"), ("aceitam revistas como doação?", "d09")]
hits, rr = 0, 0.0
for q, certo in avaliacao:
    ordem = [DOCS[i]["id"] for i in np.argsort(-bm.get_scores(re.findall(r"\w+", q.lower())))]
    pos = ordem.index(certo) + 1; hits += pos <= 3; rr += 1 / pos
print(f"hit@3 = {hits / len(avaliacao):.2f} | MRR = {rr / len(avaliacao):.2f}")

## Evaluation frameworks for RAG systems · PDF p. 446

#### 🧪 Exercício: Fidelidade com modelo-juiz

Para uma pergunta, gere a resposta do RAG e peça ao modelo forte (`MODELO_FORTE`) uma nota de 1 a 5 de **fidelidade** (a resposta só afirma o que está no contexto?) em JSON `{"nota": n, "motivo": "..."}`.

In [ ]:
ctx = DOCS[3]["texto"]
resposta = perguntar(f"Contexto: {ctx}\nPergunta: quanto custa atrasar 3 livros por 2 dias? Responda usando só o contexto.")
juiz = perguntar(f'Avalie a FIDELIDADE da resposta ao contexto (1 a 5). JSON {{"nota": n, "motivo": "..."}}.\nContexto: {ctx}\nResposta: {resposta}',
                 modelo=MODELO_FORTE, response_format={"type": "json_object"}, temperature=0)
print(resposta); print(json.loads(juiz))